# 05 · Normalize, cell cycle, HVGs and PCA

*sc-rnaseq-wf, re-instantiated step by step for `team-jakobsson` (`jakobsson-pmdbs-sn-rnaseq`)*

| | |
| :- | :- |
| WDL task | [`normalize`](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/workflows/cohort_analysis/cohort_analysis.wdl) |
| Script | [`process`](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/docker/sc_tools/scripts/main/process) → `sc-process` |

`process` turns raw counts into the representation used downstream:

1. **Keep raw counts.** `layers["counts"] = X`. scVI/scANVI model raw counts, so they are kept alongside the normalized values.
2. **Normalize.** `normalize_total(target_sum=10,000)` then `log1p`. `X` now holds log-normalized expression, used for PCA and plots.
3. **Cell cycle.** `score_genes_cell_cycle` with the Tirosh et al. S and G2/M gene lists adds `S_score`, `G2M_score` and `phase`.
4. **Highly variable genes (HVGs).** `sc.experimental.pp.highly_variable_genes(flavor="pearson_residuals", layer="counts", batch_key="batch_id", n_top_genes=3000)` ranks genes by the variance of their Pearson residuals under a negative binomial model, within each batch, so genes that differ only between batches are not favoured. If the batch key is `sample`, it is replaced by `batch_id` (per-sample HVG selection can fail).
5. **Subset and PCA.** The object is reduced to the selected genes, and a 30-component PCA (`X_pca`) is added for Harmony and the scib baseline.

`all_genes.csv` and `hvg_genes.csv` are `var` before and after the subset.

For human data, the 17 Kamath et al. dopaminergic (DA) neuron marker genes are kept even if they are not among the top HVGs, so DA subtypes can be resolved later. scanpy returns the HVG table in gene order, not ranked, so genes are selected on its `highly_variable` flag.

**Inputs**
- `${cohort_id}.filtered.h5ad`

**Outputs** (in `cfg.RUN_DIR`, named as in the WDL)
- `${cohort_id}.normalized.h5ad`
- `${cohort_id}.all_genes.csv`
- `${cohort_id}.hvg_genes.csv`

**Previous:** [04_mmc](04_mmc.ipynb) · **Next:** [06_transcriptional_phenotype](06_transcriptional_phenotype.ipynb)

In [ ]:
import wf_config as cfg
import pandas as pd
import scanpy as sc
from IPython.display import Image, display

cfg.RUN_DIR.mkdir(parents=True, exist_ok=True)
print("run directory:", cfg.RUN_DIR)

## 1. Run `sc-process`

In [ ]:
cfg.run(
    "sc-process",
    "--adata-input", cfg.out("filtered.h5ad"),
    "--workflow-name", cfg.WORKFLOW_NAME,
    "--batch-key", cfg.BATCH_KEY,
    "--norm-target-sum", cfg.NORM_TARGET_SUM,
    "--n-top-genes", cfg.N_TOP_GENES,
    "--n-comps", cfg.N_COMPS,
    "--adata-output", cfg.out("normalized.h5ad"),
    "--output-all-genes", cfg.out("all_genes.csv"),
    "--output-hvg-genes", cfg.out("hvg_genes.csv"),
)

## 2. Inspect the normalized object

In [ ]:
adata = sc.read_h5ad(cfg.out("normalized.h5ad"))
adata

In [ ]:
all_genes = pd.read_csv(cfg.out("all_genes.csv"), index_col=0)
hvg_genes = pd.read_csv(cfg.out("hvg_genes.csv"), index_col=0)
print(f"{len(hvg_genes):,} of {len(all_genes):,} genes kept")
print("cell cycle phases:")
adata.obs["phase"].value_counts()

In [ ]:
sc.pl.pca_variance_ratio(adata, n_pcs=cfg.N_COMPS, log=True)
sc.pl.pca(adata, color=["batch_id", "total_counts", "pct_counts_mt"], ncols=3)

## 3. Compare the HVGs with the release

In [ ]:
released_hvgs = cfg.curated("hvg_genes.csv")
if released_hvgs.exists():
    released = set(pd.read_csv(released_hvgs, index_col=0).index)
    ours = set(hvg_genes.index)
    print(f"release {len(released):,} HVGs, this run {len(ours):,}; shared {len(ours & released):,}")
else:
    print(f"{released_hvgs.name} not in the curated release")

**Previous:** [04_mmc](04_mmc.ipynb) · **Next:** [06_transcriptional_phenotype](06_transcriptional_phenotype.ipynb)